# ML-07 — Baseline Action Score and Top-20 Review

## 1. My rule and its reason codes

In [1]:
import pandas as pd, numpy as np
d = pd.read_csv("../outputs/feature_vector.csv")
BASE = d["future_decline"].mean()
FLOOR = 50
print(f"rows {len(d):,} | decline base rate {BASE:.3f}\n")

d["freshness"] = pd.cut(d.days_since_last_update, [-1, 30, 90, 180, 100000],
                        labels=["0-30", "31-90", "91-180", "181+"])
gA = d.groupby("freshness", observed=True)["future_decline"].agg(decline_rate="mean", n="count").round(3)
print("Signal A - recency (behind the refresh flag):")
print(gA[gA.n >= FLOOR].to_string())

d["ctr_q"] = pd.qcut(d.ctr_prev_30d.rank(method="first"), 4, labels=["low", "mid-low", "mid-high", "high"])
gB = d.groupby("ctr_q", observed=True)["future_decline"].agg(decline_rate="mean", n="count").round(3)
print("\nSignal B - historical CTR (behind the CTR-fix logic):")
print(gB[gB.n >= FLOOR].to_string())

rows 145,030 | decline base rate 0.544

Signal A - recency (behind the refresh flag):
           decline_rate      n
freshness                     
0-30              0.346  10423
31-90             0.551  45046
91-180            0.576  15554
181+              0.561  74007

Signal B - historical CTR (behind the CTR-fix logic):
          decline_rate      n
ctr_q                        
low              0.569  36258
mid-low          0.572  36257
mid-high         0.580  36257
high             0.455  36258


**Two signal checks behind the rule** (base ≈ 0.544; n shown, floor 50):

- **Signal A — recency (behind the *refresh* flag).** The refresh flag assumes stale pages are at risk. Pages updated in the last **0–30 days decline 0.346** vs **0.561 at 181+ days**. **Verdict: CONFIRMED** — recent updates are protective, so "not recently updated" earns a risk point (`stale_content`).
- **Signal B — historical CTR (behind the *CTR-fix* logic).** The CTR-fix logic assumes thinly-clicked pages are at risk. The **top-CTR quartile declines 0.455** vs **~0.57** for the lower three. **Verdict: CONFIRMED** — weak engagement is a risk direction, so low CTR earns a point (`thin_ctr`).

Falling momentum is a weaker third signal (MIXED in the deeper ML-06 audit) but directionally consistent, so it earns the third point. Content **age** is deliberately left out — ML-06 showed decline is non-monotonic in age, so it is not a clean risk direction.

**The rule, in three sentences.** A page goes to the top of the review queue when it shows the risk signals the ML-06 audit actually supported on the real forward label: its **engagement (CTR) is thin**, it has **not been refreshed recently** (≥ 31 days since last update), and it is **losing historical momentum**. Each condition that fires adds one point (0–3), so the score is a plain count anyone can re-derive by hand — no fitted weights. Among pages with the same score, the one with **lower historical CTR** ranks first (the cleanest single signal).

**Reason codes** (each is one point, emitted only when true):

| Reason code | Fires when | Audit backing (ML-06) |
|---|---|---|
| `thin_ctr` | historical CTR below the slice median | S3 CONFIRMED (top-CTR pages are the steadiest) |
| `stale_content` | ≥ 31 days since last update | freshness flag CONFIRMED at the fresh end |
| `losing_momentum` | `hist_impr_momentum < -0.05` (Feb→Mar falling) | S1 (weak/directional) |

I deliberately dropped content **age** from the rule: ML-06 showed decline is non-monotonic in age, so age is not a clean risk direction. Thresholds are **frozen** and re-used unchanged when the model is compared against this baseline in ML-08. The score never touches the April label window or any product flag — leakage-checked in §4.

## 2. Build the ranked queue (writes the CSV)

The queue scores **every** page, ranks it, and writes `work/outputs/baseline_action_score.csv`. The honest metric is **precision@K** — of the top K flagged, how many actually declined — read next to the base rate and the majority/dummy floor (both equal the decline rate). On a genuinely hard forward-prediction task, expect the numbers to sit **close to the base rate**; the value is in the clean low-risk floor and in the decision-relevant (`reach ≥ 100`) slice.

In [2]:
import pandas as pd, numpy as np
d = pd.read_csv("../outputs/feature_vector.csv")
BASE = d["future_decline"].mean()

MED_CTR = d.ctr_prev_30d.median()

def rule_score(df):
    cond = {
        "thin_ctr":        df.ctr_prev_30d < MED_CTR,
        "stale_content":   df.days_since_last_update >= 31,
        "losing_momentum": df.hist_impr_momentum < -0.05,
    }
    pts = sum(c.astype(int) for c in cond.values()).values
    mat = pd.DataFrame({k: v.values for k, v in cond.items()})
    codes = mat.apply(lambda r: "|".join(mat.columns[r.values]) or "none", axis=1).values
    return pts, codes

d["risk_points"], d["reason_codes"] = rule_score(d)
d["ctr_rank"] = d.ctr_prev_30d.rank(pct=True)
d["reach"] = d.reach_impr
d["action"] = np.select([d.risk_points >= 2, d.risk_points == 1],
                        ["Priority review", "Review"], default="Monitor / deprioritize")
d["confidence"] = np.select([d.risk_points >= 2, d.risk_points == 1],
                            ["higher", "medium"], default="low")

d = d.sort_values(["risk_points", "ctr_rank", "content_id"], ascending=[False, True, True]).reset_index(drop=True)
d["rank"] = np.arange(1, len(d) + 1)

out_cols = ["rank","content_id","client_id","risk_points","reason_codes","reach",
            "action","confidence","future_decline"]
d[out_cols].to_csv("../outputs/baseline_action_score.csv", index=False)
print("wrote ../outputs/baseline_action_score.csv ->", d[out_cols].shape)

reach_mask = d.reach.values >= 100
for k in [20, 50, 100]:
    print(f"precision@{k}: {d.future_decline.iloc[:k].mean():.3f}   (base rate {BASE:.3f})")
sub = d[reach_mask]
print(f"precision@50 on reach>=100 slice: {sub.future_decline.iloc[:50].mean():.3f}  (base {sub.future_decline.mean():.3f})")
print(f"majority / dummy baseline precision = base rate = {BASE:.3f}")

wrote ../outputs/baseline_action_score.csv -> (145030, 9)
precision@20: 0.600   (base rate 0.544)
precision@50: 0.560   (base rate 0.544)
precision@100: 0.620   (base rate 0.544)
precision@50 on reach>=100 slice: 0.660  (base 0.538)
majority / dummy baseline precision = base rate = 0.544


## 3. Top-20 review

In [3]:
show = d.head(20).copy()
show["cid"] = show.content_id.str[-6:]
print("TOP 20 review queue:")
print(show[["rank","cid","risk_points","reason_codes","reach","action","future_decline"]]
      .to_string(index=False))
print("\ndecline rate by risk_points (the low-risk floor is the clean result):")
print(d.groupby("risk_points")["future_decline"].agg(decline_rate="mean", n="count").round(3).to_string())

TOP 20 review queue:
 rank    cid  risk_points                  reason_codes  reach          action  future_decline
    1 bfb4a8            2 stale_content|losing_momentum     15 Priority review               1
    2 26c7db            2 stale_content|losing_momentum    136 Priority review               1
    3 70fe14            2 stale_content|losing_momentum     10 Priority review               1
    4 f6e376            2 stale_content|losing_momentum     34 Priority review               1
    5 cdf58b            2 stale_content|losing_momentum     80 Priority review               0
    6 b1ba45            2 stale_content|losing_momentum      2 Priority review               0
    7 2b9e9d            2 stale_content|losing_momentum    100 Priority review               1
    8 ad346e            2 stale_content|losing_momentum      2 Priority review               1
    9 56ca70            2 stale_content|losing_momentum      1 Priority review               0
   10 ffa87e            2 sta

**Top-20 review.** The very top of the list is pages that fire the most reason codes and sit at the bottom of the CTR distribution: **thinly-engaged, not-recently-refreshed, drifting** pages.

- **Action:** `Priority review` for the top band.
- **Confidence:** `directional` at best. On this real forward label the top-of-list precision hovers only a little above the 0.544 base rate overall (it separates more on the `reach ≥ 100` slice) — so treat a high rank as "worth a look," never "this will drop."
- **What would make a pick wrong:** (a) a thin-CTR page that is *stable*, not declining — low engagement is a level, not a trend; (b) a page whose "stale" flag is fine because the topic is evergreen and needs no refresh; (c) a small page whose momentum is just noise. The hand-review below finds plenty of these — precision near the base rate means **roughly half the top picks are false alarms.**

## 4. Weak picks + leakage check

In [4]:
top50 = d.head(50)
weak = top50[top50.future_decline == 0].copy()
weak["cid"] = weak.content_id.str[-6:]
print(f"weak picks in top 50: {len(weak)} of 50 were wrong (precision@50 = {top50.future_decline.mean():.3f})")
print(weak[["rank","cid","risk_points","reason_codes","reach"]].head(10).to_string(index=False))

SCORING_INPUTS = {"ctr_prev_30d", "days_since_last_update", "hist_impr_momentum"}
FORBIDDEN = {"impr_future","future_decline","gsc_avg_position","health_score",
             "content_updated_date","provider_used","model_used","content_id","client_id"}
feat_cols = set(pd.read_csv("../outputs/feature_vector.csv").columns)
assert not (SCORING_INPUTS & FORBIDDEN), "product flag / future window leaked into the score"
assert SCORING_INPUTS <= feat_cols, "score reads a column outside the feature vector"
print("\nleakage check passed: score inputs", sorted(SCORING_INPUTS))
print("  no label window, no product flag, no identifier.")

weak picks in top 50: 22 of 50 were wrong (precision@50 = 0.560)
 rank    cid  risk_points                  reason_codes  reach
    5 cdf58b            2 stale_content|losing_momentum     80
    6 b1ba45            2 stale_content|losing_momentum      2
    9 56ca70            2 stale_content|losing_momentum      1
   12 cc9afe            2 stale_content|losing_momentum      1
   13 74cb07            2 stale_content|losing_momentum     49
   15 2fa60f            2 stale_content|losing_momentum     35
   16 fad551            2 stale_content|losing_momentum      3
   20 98c909            2 stale_content|losing_momentum   2270
   22 eed85f            2 stale_content|losing_momentum     18
   23 4d8747            2 stale_content|losing_momentum    160



leakage check passed: score inputs ['ctr_prev_30d', 'days_since_last_update', 'hist_impr_momentum']
  no label window, no product flag, no identifier.


**Weak picks.** Around half of the top 50 did **not** decline — visible above, and expected: with a base rate of ~0.54 and weak signals, a transparent rule cannot do much better than a coin-flip-plus at the very top. The one genuinely clean result is the **low-risk floor**: pages that fire **zero** reason codes decline at only ~0.33, well below base — the rule is better at saying "leave this alone" than "this will drop." That asymmetry is itself useful for triage.

**Leakage check (passed).** The rule reads only three history-only columns — `ctr_prev_30d`, `days_since_last_update`, `hist_impr_momentum` — all from the ML-05 warehouse feature vector. The cell asserts the scoring inputs have **empty intersection** with the forbidden set (the April label window, any product flag/health score, identifiers) and are a subset of the saved matrix. No future window and no decision-derived flag feeds the score.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.